In [0]:
dbutils.library.restartPython()

In [0]:
import sys
import uuid

from pathlib import Path
from pyspark.sql import functions as F

CODE_ROOT = "/Volumes/workspace/retail_dev_schema/retail_dev_dbx_vol"

if CODE_ROOT not in sys.path:
    sys.path.insert(0, CODE_ROOT)

from utils.pipeline_config import (
    get_config,
    load_contract,
    ENTITIES,
)
from scripts.py_extractor import extract_delivery
from scripts.bronze_loader import run_historical_load

ENVIRONMENT = "dev"
TRIGGER_TYPE = "MANUAL"

# Use the existing snapshot for the first test.
EXTRACT_NEW = False
DELIVERY_NAME = "batch_003"

# Enable only to intentionally replay under changed YAML.
REPROCESS_CONTRACT = False

config = get_config(ENVIRONMENT)
spark.conf.set("spark.sql.session.timeZone", "UTC")

In [0]:
run_id = str(uuid.uuid4())

contract = load_contract(
    spark,
    f"../config/source_contracts.yaml",
)

if EXTRACT_NEW:
    delivery_folder = extract_delivery(
        config,
        DELIVERY_NAME,
    )
else:
    delivery_folder = (
        Path(config.landing_root) / DELIVERY_NAME
    )

print("Run ID:", run_id)
print("Delivery:", delivery_folder)
print("Contract version:", contract["version"])
print("Contract checksum:", contract["checksum"])

In [0]:
results = run_historical_load(
    spark=spark,
    config=config,
    delivery_folder=delivery_folder,
    run_id=run_id,
    contract=contract,
    trigger_type=TRIGGER_TYPE,
    reprocess_contract=REPROCESS_CONTRACT,
)

display(spark.createDataFrame(results))

In [0]:
display(
    spark.table(config.monitoring_table)
    .filter(F.col("ingestion_run_id") == run_id)
    .select(
        "batch_id",
        "entity",
        "ingestion_run_id",
        "trigger_type",
        "status",
        "contract_version",
        "expected_count",
        "input_count",
        "accepted_count",
        "rejected_count",
        "failed_step",
        "skipped_reason",
        "error_message",
    )
    .orderBy("entity")
)

In [0]:
for entity in ENTITIES:
    print(entity)

    display(
        spark.table(config.bronze_table(entity))
        .groupBy("batch_id")
        .count()
        .orderBy("batch_id")
    )

In [0]:
display(
    spark.table(config.monitoring_table)
    .select(
        "started_at",
        "entity",
        "batch_id",
        "ingestion_run_id",
        "trigger_type",
        "status",
        "accepted_count",
        "rejected_count",
        "original_success_run_id",
    )
    .orderBy("started_at", "entity")
)